In [ ]:
%%capture

%pip install qbraid qiskit stimcirq ply matplotlib

In [ ]:
import warnings
import logging

logger = logging.getLogger()

logger.setLevel(logging.INFO)

warnings.filterwarnings("ignore", category=UserWarning)

Create our qiskit circuit

In [ ]:
from qiskit import QuantumCircuit

In [ ]:
qiskit_circuit = QuantumCircuit(2, 2)

qiskit_circuit.h(0)
qiskit_circuit.cx(0, 1)
qiskit_circuit.measure_all()

qiskit_circuit.draw()

Now let's look at our currrent conversion graph

In [ ]:
from qbraid.programs import QPROGRAM_REGISTRY
from qbraid.transpiler import ConversionGraph

In [ ]:
QPROGRAM_REGISTRY

In [ ]:
graph = ConversionGraph()

In [ ]:
graph.plot()

To convert our qiskit circuit to stim, we need to define a new conversion

In [ ]:
from qbraid.transpiler import Conversion

try:
    from stimcirq import cirq_circuit_to_stim_circuit
except ImportError:
    cirq_circuit_to_stim_circuit = None
    print("stimcirq is not installed. Install via: pip install stimcirq")
    print("Skipping remaining conversion steps.")

In [ ]:
if cirq_circuit_to_stim_circuit is not None:
    conversion = Conversion("cirq", "stim", cirq_circuit_to_stim_circuit)

In [ ]:
if cirq_circuit_to_stim_circuit is not None:
    try:
        graph.add_conversion(conversion, overwrite=True)
    except Exception:
        # Workaround for SDK graph edge inconsistency
        graph._conversions = [c for c in graph._conversions if not (c.source == "cirq" and c.target == "stim")]
        graph.add_conversion(conversion)

In [ ]:
graph.plot()

In [ ]:
graph.has_path("qiskit", "stim")

Now we have a path to convert from qiskit to stim!

In [ ]:
from qbraid.transpiler import transpile

Perform the conversion using our custom conversion graph

In [ ]:
if cirq_circuit_to_stim_circuit is not None:
    stim_circuit = transpile(qiskit_circuit, "stim", conversion_graph=graph)
    print(type(stim_circuit))
else:
    print("Skipped: stimcirq not available.")

Sample the stim circuit

In [ ]:
if cirq_circuit_to_stim_circuit is not None:
    stim_circuit.diagram()
else:
    print("Skipped: stimcirq not available.")

In [ ]:
if cirq_circuit_to_stim_circuit is not None:
    sampler = stim_circuit.compile_sampler()
    print(sampler.sample(shots=10))
else:
    print("Skipped: stimcirq not available.")